# AI Subscription Manager — Agentic AI Demo

This notebook demonstrates the three required course outcomes:

1. **Tool use:** the agent calls tools instead of only generating text.
2. **Multi-step behavior:** the agent uses tool results to decide what to do next.
3. **Memory:** information from an earlier turn is stored and used in a later turn.

This notebook uses the project's existing `src/agent.py`, `src/tools.py`, and `src/database.py`.

## Run instructions

Run this notebook with the **same `venv` kernel** used by the project.

Open the notebook from:

```text
subscription-manager-agent/
└── notebooks/
    └── agent_demo.ipynb
```

The first code cell automatically finds the project root by looking for `src/database.py`.

Your `.env` file must contain:

```text
GROQ_API_KEY=your_groq_api_key_here
```

The demo uses separate SQLite files so it does not overwrite your normal application database.

In [1]:
import sys
from pathlib import Path

# Find the project root by locating src/database.py.
PROJECT_ROOT = None

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "src" / "database.py").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the project root. Open the notebook inside the project."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import database

# Use a separate database for this notebook demo.
database.DB_PATH = PROJECT_ROOT / "demo_subscriptions.db"

if database.DB_PATH.exists():
    database.DB_PATH.unlink()

database.initialize_database()

from agents import Runner, SQLiteSession, ToolCallItem, ToolCallOutputItem
from agent import subscription_agent

# Separate conversation-memory database for the notebook.
session_db = PROJECT_ROOT / "demo_agent_memory.db"

if session_db.exists():
    session_db.unlink()

session = SQLiteSession(
    "subscription_manager_course_demo",
    db_path=session_db,
)

print("Demo environment ready.")
print("Project root:", PROJECT_ROOT)
print("Demo database:", database.DB_PATH)

Demo environment ready.
Project root: d:\CODEPLAY\subscription-manager-agent
Demo database: d:\CODEPLAY\subscription-manager-agent\demo_subscriptions.db


## Multi-step trace

The helper below prints the actual tool calls and tool results returned by the agent run. This is the evidence that the system is using tools and performing multiple steps.

In [2]:
async def run_with_trace(goal):
    print("=" * 80)
    print("USER GOAL")
    print("=" * 80)
    print(goal)

    result = await Runner.run(
        subscription_agent,
        goal,
        session=session,
    )

    print("\n" + "=" * 80)
    print("MULTI-STEP TRACE")
    print("=" * 80)

    step = 1

    for item in result.new_items:
        if isinstance(item, ToolCallItem):
            tool_name = getattr(item, "tool_name", "unknown_tool")
            raw_item = getattr(item, "raw_item", None)
            arguments = getattr(raw_item, "arguments", "")

            print(f"Step {step}: TOOL CALL")
            print(f"  Tool: {tool_name}")
            print(f"  Arguments: {arguments}")
            step += 1

        elif isinstance(item, ToolCallOutputItem):
            print(f"Step {step}: TOOL RESULT")
            print(f"  Result: {item.output}")
            step += 1

    print("\n" + "=" * 80)
    print("FINAL AGENT RESPONSE")
    print("=" * 80)
    print(result.final_output)

    return result

# Goal 1 — Create subscription state

The user gives the agent a monthly budget and three subscriptions.

This turn establishes persistent application state that later turns can retrieve.

In [3]:
await run_with_trace(
    "Set my monthly subscription budget to ₹2000 and add these subscriptions: "
    "Netflix for ₹649 per month, renewing on 2026-09-01; "
    "Spotify for ₹1199 per month, renewing on 2026-09-05; "
    "Amazon Prime for ₹299 per month, renewing on 2026-09-15."
)

USER GOAL
Set my monthly subscription budget to ₹2000 and add these subscriptions: Netflix for ₹649 per month, renewing on 2026-09-01; Spotify for ₹1199 per month, renewing on 2026-09-05; Amazon Prime for ₹299 per month, renewing on 2026-09-15.

MULTI-STEP TRACE
Step 1: TOOL CALL
  Tool: set_monthly_budget
  Arguments: {"budget":2000}
Step 2: TOOL RESULT
  Result: Monthly subscription budget set to ₹2000.00.
Step 3: TOOL CALL
  Tool: add_subscription
  Arguments: {"cost":649,"name":"Netflix","renewal_date":"2026-09-01"}
Step 4: TOOL RESULT
  Result: Subscription 'Netflix' was added successfully. Monthly cost: ₹649.00. Renewal date: 2026-09-01.
Step 5: TOOL CALL
  Tool: add_subscription
  Arguments: {"cost":1199,"name":"Spotify","renewal_date":"2026-09-05"}
Step 6: TOOL RESULT
  Result: Subscription 'Spotify' was added successfully. Monthly cost: ₹1199.00. Renewal date: 2026-09-05.
Step 7: TOOL CALL
  Tool: add_subscription
  Arguments: {"cost":299,"name":"Amazon Prime","renewal_date":"

RunResult(input=[{'content': 'Set my monthly subscription budget to ₹2000 and add these subscriptions: Netflix for ₹649 per month, renewing on 2026-09-01; Spotify for ₹1199 per month, renewing on 2026-09-05; Amazon Prime for ₹299 per month, renewing on 2026-09-15.', 'role': 'user'}], new_items=[ReasoningItem(agent=Agent(name='Subscription Manager', handoff_description=None, tools=[FunctionTool(name='add_subscription', description='Add a new active subscription.', params_json_schema={'properties': {'name': {'description': 'Name of the subscription.', 'title': 'Name', 'type': 'string'}, 'cost': {'description': 'Monthly cost in rupees.', 'title': 'Cost', 'type': 'number'}, 'renewal_date': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Renewal date in YYYY-MM-DD format, if known.', 'title': 'Renewal Date'}}, 'required': ['name', 'cost', 'renewal_date'], 'title': 'add_subscription_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Failur

### State created

The database should now contain:

```text
Budget: ₹2000

Netflix       ₹649/month
Spotify       ₹1199/month
Amazon Prime  ₹299/month
```

The next goal will not repeat these details.

# Goal 2 — Agentic cancellation recommendation

Ask the agent:

> What subscription should I consider cancelling to get back under budget?

The agent needs to obtain the budget, monthly total, and active subscriptions before deciding what to recommend.

A typical multi-step trace is:

```text
get_monthly_budget
        ↓
get_monthly_total
        ↓
get_subscriptions
        ↓
reason over tool results
        ↓
cancellation recommendation
```

The exact tool order can vary because the agent decides what to do next.

In [4]:
await run_with_trace(
    "What subscription should I consider cancelling to get back under budget?"
)

USER GOAL
What subscription should I consider cancelling to get back under budget?

MULTI-STEP TRACE
Step 1: TOOL CALL
  Tool: get_monthly_budget
  Arguments: {"request":"user wants to see current budget"}
Step 2: TOOL RESULT
  Result: Current monthly subscription budget: ₹2000.00
Step 3: TOOL CALL
  Tool: get_monthly_total
  Arguments: {"request":"calculate current spending"}
Step 4: TOOL RESULT
  Result: Current monthly subscription cost: ₹2147.00
Step 5: TOOL CALL
  Tool: get_subscriptions
  Arguments: {"request":"user wants to see subscriptions"}
Step 6: TOOL RESULT
  Result: Netflix: ₹649.00/month, renewal date: 2026-09-01
Spotify: ₹1199.00/month, renewal date: 2026-09-05
Amazon Prime: ₹299.00/month, renewal date: 2026-09-15

FINAL AGENT RESPONSE
You’re currently spending **₹2147** per month on subscriptions, which is **₹147** over your ₹2000 budget.  
To bring your spending back under budget while keeping as many services as possible, the best option is:

| Recommendation | Subsc

RunResult(input=[{'content': 'Set my monthly subscription budget to ₹2000 and add these subscriptions: Netflix for ₹649 per month, renewing on 2026-09-01; Spotify for ₹1199 per month, renewing on 2026-09-05; Amazon Prime for ₹299 per month, renewing on 2026-09-15.', 'role': 'user'}, {'id': '__fake_id__', 'summary': [], 'type': 'reasoning', 'content': [{'text': 'We need to set monthly budget to 2000, and add three subscriptions.\n\nWe must call set_monthly_budget with budget=2000. Then add_subscription thrice.\n\nBut we need to use tools via JSON arguments. We should do sequential calls.\n\nWe need to respond with confirmation after each. Probably we should call set_monthly_budget first, then add each subscription. Provide each call separately.\n\nWe also might want to use the tool set_monthly_budget first, then add_subscription for each.\n\nWe need to produce tool calls. Since we can call multiple. The instruction: "When calling a tool, always provide valid JSON arguments that match th

### Why this is agentic

The current state is:

```text
Monthly total = ₹2147
Budget = ₹2000
Over budget = ₹147
```

Possible single-subscription results include:

```text
Netflix:       ₹2147 - ₹649  = ₹1498
Spotify:       ₹2147 - ₹1199 = ₹948
Amazon Prime:  ₹2147 - ₹299  = ₹1848
```

The agent can recommend Amazon Prime because it brings the estimated monthly total below the budget.

The agent only **recommends** a cancellation; it does not automatically cancel anything.

# Goal 3 — Use remembered information later

Now ask a different question without repeating the subscription names or prices.

This demonstrates that information established earlier can be used later.

In [5]:
await run_with_trace(
    "How much will my current subscriptions cost per year?"
)

USER GOAL
How much will my current subscriptions cost per year?

MULTI-STEP TRACE
Step 1: TOOL CALL
  Tool: get_yearly_cost
  Arguments: {"request":"user wants to see yearly subscription cost"}
Step 2: TOOL RESULT
  Result: Current monthly cost: ₹2147.00. Estimated yearly cost: ₹25764.00.

FINAL AGENT RESPONSE
Your current subscriptions total **₹2147 per month**, which adds up to an estimated yearly cost of **₹25,764**. If you have any questions about this or need help adjusting your plan, just let me know!


RunResult(input=[{'content': 'Set my monthly subscription budget to ₹2000 and add these subscriptions: Netflix for ₹649 per month, renewing on 2026-09-01; Spotify for ₹1199 per month, renewing on 2026-09-05; Amazon Prime for ₹299 per month, renewing on 2026-09-15.', 'role': 'user'}, {'id': '__fake_id__', 'summary': [], 'type': 'reasoning', 'content': [{'text': 'We need to set monthly budget to 2000, and add three subscriptions.\n\nWe must call set_monthly_budget with budget=2000. Then add_subscription thrice.\n\nBut we need to use tools via JSON arguments. We should do sequential calls.\n\nWe need to respond with confirmation after each. Probably we should call set_monthly_budget first, then add each subscription. Provide each call separately.\n\nWe also might want to use the tool set_monthly_budget first, then add_subscription for each.\n\nWe need to produce tool calls. Since we can call multiple. The instruction: "When calling a tool, always provide valid JSON arguments that match th

### Expected calculation

```text
₹2147 × 12 = ₹25764
```

The user did not provide the subscription prices again. The agent uses the stored subscription state.